# 01. KuaiRec Data Preparation

This notebook rebuilds the first data-preparation step for the RecSys research project. Its job is to produce one clean interaction-level table that later notebooks can use for GNN training, belief simulation, structural estimation, and semi-synthetic counterfactual experiments.

Dataset reference: https://kuairec.com/

The guiding principle is **provenance first**: every column should be traceable to a raw file or to a clearly documented transformation. Future me should be able to reopen this notebook years later and understand which variables are pre-exposure features, which are post-exposure outcomes, which are shifted history variables, and which columns are only descriptors for auditing.

Final output:

- `KuaiRec 2.0/data/processed/processed_data.parquet`
- optional: `KuaiRec 2.0/data/processed/processed_data.csv`

Important design choices:

- Session IDs are created from one-hour gaps within each user history.
- The three dense activity bursts are kept; interactions outside them are dropped.
- Engagement heads (`y_complete`, `y_long`, `y_rewatch`, `y_neg`) are observed outcomes, not predictions.
- History features are shifted so they only use information available before the current interaction.
- Context time features are derived from the parsed interaction `time` column, not from the ambiguous raw `timestamp` column.
- Same-day item aggregate counters from `item_daily_features.csv` are intentionally excluded because they summarize other users' realized behavior on the same day and can leak post-exposure information.

## Complete Feature Dictionary

This table lists the columns created or preserved by the notebook. The `Serving-safe?` column means whether the variable can be used as an inference-time feature for predicting a response before the current recommendation outcome is observed. Columns marked `No` may still be saved because later notebooks need them as labels, audit fields, or grouping keys.

| Group | Feature(s) | Source or construction | Meaning | Serving-safe? |
|---|---|---|---|---|
| Raw keys | `user_id`, `video_id` | `big_matrix.csv` | User and video identifiers. | Yes as IDs/keys; use carefully in out-of-time validation. |
| Raw time | `time`, `date`, `timestamp` | `big_matrix.csv` | Interaction time. `time` is the canonical parsed timestamp. | Yes if recommendation time is known. |
| Raw outcome fields | `play_duration`, `watch_ratio` | `big_matrix.csv` | Realized watch behavior after exposure. KuaiRec describes `watch_ratio` as the interaction label. | No. Labels/audit only. |
| Burst/session descriptors | `burst_id`, `session`, `sess_rank` | Derived from `time` and one-hour inactivity gaps. | Dense-window ID, cumulative user session ID, and position inside session. | Mostly yes; they use past/current time only. |
| Full-session descriptor | `sess_len` | Derived after grouping the completed session. | Final number of interactions in the session. | No for live serving; future session length is unknown during the session. |
| Observed heads | `y_complete`, `y_long`, `y_rewatch`, `y_neg` | Mathematical functions of `play_duration` and `watch_ratio`. | Binary realized engagement outcomes used as labels. | No. Labels only. |
| User static categorical | `u_user_active_degree`, `u_follow_user_num_range`, `u_fans_user_num_range`, `u_friend_user_num_range`, `u_register_days_range` | `user_features.csv` | Platform-provided user state and binned counts. | Yes, assuming the snapshot is available before serving. |
| User static binary | `u_is_lowactive_period`, `u_is_live_streamer`, `u_is_video_author` | `user_features.csv` | Binary user attributes. | Yes, assuming the snapshot is available before serving. |
| User static numeric | `u_follow_user_num`, `u_fans_user_num`, `u_friend_user_num`, `u_register_days` | `user_features.csv` | User social graph counts and account age. | Yes, assuming the snapshot is available before serving. |
| User numeric transforms | `u_follow_user_num_log1p`, `u_fans_user_num_log1p`, `u_friend_user_num_log1p`, `u_register_days_log1p` | `log1p` transforms of numeric user counts. | Less-skewed versions of the raw count variables. | Yes, same assumption as raw user counts. |
| User encrypted tags | `u_onehot_feat0` to `u_onehot_feat17` | `user_features.csv` | Encrypted categorical user segments. | Yes, assuming the snapshot is available before serving. |
| Item static categorical | `i_author_id`, `i_video_type`, `i_upload_type`, `i_visible_status`, `i_music_id`, `i_video_tag_id`, `i_video_tag_name` | Stable columns from `item_daily_features.csv`; one deterministic row kept per video. | Author, type, upload source, visibility, music, and tag metadata. | Mostly yes; `i_visible_status` is snapshot-like, so treat cautiously in strict historical tests. |
| Item static numeric | `i_video_duration`, `i_aspect_ratio`, `i_age_since_upload_days` | `item_daily_features.csv`, raw duration fallback, and interaction time. | Video length, width/height, and item age at interaction. | Yes. |
| Item category hierarchy | `i_cat_level1_id`, `i_cat_level1_name`, `i_cat_level2_id`, `i_cat_level2_name`, `i_cat_level3_id`, `i_cat_level3_name` | `kuairec_caption_category.csv` | Caption-derived content taxonomy. | Yes as item metadata. |
| Context | `ctx_hour_sin`, `ctx_hour_cos`, `ctx_is_weekend` | Derived from `time`. | Time-of-day cycle and weekend indicator. | Yes. |
| User history EMA | `hist_ema_y_complete`, `hist_ema_y_long`, `hist_ema_y_rewatch`, `hist_ema_y_neg`, `hist_ema_watchratio` | Per-user EWMAs shifted by one interaction. | User's prior response tendency. | Yes; uses only previous outcomes. |
| User-category history | `hist_cat_ema_complete`, `hist_cat_entropy_l2` | Shifted user/category completion EMA and entropy of previous level-2 categories. | Prior category-specific response and historical content diversity. | Yes; uses only previous interactions. |
| User-author history | `hist_author_recency_days`, `hist_last_complete_author`, `hist_has_author_history` | Shifted within user x burst x author. | Prior exposure/response to the same author in the current burst. | Yes; uses only previous author interactions. |
| Previous-session history | `hist_prev_sess_len`, `hist_intersess_gap_h` | Previous completed session within the same burst. | Prior session size and time since prior session ended. | Yes; uses only earlier completed sessions. |

## Leakage Rule

For serving-time ML models, do not use `play_duration`, `watch_ratio`, `y_*`, or `sess_len` as features. Also do not add item daily aggregate fields such as `show_cnt`, `play_cnt`, `complete_play_cnt`, `like_cnt`, `follow_cnt`, `share_cnt`, `report_cnt`, or their user-count variants. Those aggregate fields summarize realized behavior and can contain information from the same day or later than the target exposure.


## Table of Contents

1. Configure paths and constants.
2. Load the raw interaction table.
3. Keep the three KuaiRec bursts, define sessions, and create observed engagement labels.
4. Merge user static features.
5. Merge item static features and category metadata.
6. Create context features.
7. Create history features that only use past information.
8. Validate the final table and save it.

The notebook is intentionally verbose. The comments are part of the research record, not just programming notes.


## 0. Setup

This section defines paths and constants. Keep all knobs here so later robustness checks are easy to find.


In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

PROJECT_ROOT = Path("/Users/haozhangao/Desktop/RecSys Research")
DATA_DIR = PROJECT_ROOT / "KuaiRec 2.0" / "data"
OUT_DIR = DATA_DIR / "processed"
OUT_DIR.mkdir(parents=True, exist_ok=True)

RAW_INTERACTIONS_PATH = DATA_DIR / "big_matrix.csv"
USER_FEATURES_PATH = DATA_DIR / "user_features.csv"
ITEM_DAILY_FEATURES_PATH = DATA_DIR / "item_daily_features.csv"
CAPTION_CATEGORY_PATH = DATA_DIR / "kuairec_caption_category.csv"

OUTPUT_PARQUET = OUT_DIR / "processed_data.parquet"
OUTPUT_CSV = OUT_DIR / "processed_data.csv"

# Set to True only when you really need a CSV copy. The CSV is large and slow to write.
WRITE_CSV = False

# The paper focuses on the three dense windows in KuaiRec.
BURST_RANGES = [
    ("2020-07-05", "2020-07-12"),
    ("2020-08-01", "2020-08-10"),
    ("2020-08-27", "2020-09-05"),
]

# A new session begins after this many seconds without an interaction.
SESSION_GAP_SECONDS = 3600

# Observed engagement-head definitions.
COMPLETE_WATCH_RATIO = 1.0
LONG_WATCH_RATIO = 1.5
LONG_PLAY_MS = 12_000
REWATCH_WATCH_RATIO = 2.0
NEGATIVE_PLAY_MS = 2_000

UNKNOWN_LEVEL1_CATEGORY_ID = -124
UNKNOWN_CATEGORY_NAME = "UNKNOWN"

required_paths = [
    RAW_INTERACTIONS_PATH,
    USER_FEATURES_PATH,
    ITEM_DAILY_FEATURES_PATH,
    CAPTION_CATEGORY_PATH,
]
missing_paths = [str(p) for p in required_paths if not p.exists()]
if missing_paths:
    raise FileNotFoundError("Missing required raw data files:\n" + "\n".join(missing_paths))

print("Project root:", PROJECT_ROOT)
print("Output parquet:", OUTPUT_PARQUET)


## 1. Load Raw Interactions

`big_matrix.csv` is the interaction log. Each row is one exposed user-video interaction with realized watch behavior. The KuaiRec documentation defines `watch_ratio = play_duration / video_duration` and notes that `watch_ratio` can be treated as the interaction label.

| Feature | Type | Description | Leakage status |
|---|---|---|---|
| `user_id` | ID | User identifier. | Safe as a key; be careful with validation splits. |
| `video_id` | ID | Viewed video identifier. | Safe as a key; be careful with validation splits. |
| `time` | datetime | Human-readable interaction timestamp. | Safe; known at recommendation time. |
| `date` | integer date | Calendar date in `YYYYMMDD` format. | Safe; known at recommendation time. |
| `timestamp` | float timestamp | Unix timestamp from the raw table. | Safe but secondary; `time` is clearer. |
| `video_duration` | milliseconds | Length of the video. Later renamed to `i_video_duration`. | Safe; item property. |
| `play_duration` | milliseconds | Realized watch time after exposure. | Not safe as a feature; outcome/audit only. |
| `watch_ratio` | ratio | Realized watch share after exposure. | Not safe as a feature; label/audit only. |

Later notebooks should be careful not to confuse observed `y_*` columns with predicted GNN heads `yhat_*`.


In [ ]:
raw_required_cols = [
    "user_id", "video_id", "play_duration", "video_duration",
    "time", "date", "timestamp", "watch_ratio",
]

print("Reading", RAW_INTERACTIONS_PATH)
df = pd.read_csv(RAW_INTERACTIONS_PATH, low_memory=False)

missing = [c for c in raw_required_cols if c not in df.columns]
if missing:
    raise KeyError(f"Raw interaction table is missing required columns: {missing}")

# Parse the human-readable interaction time. This is the canonical interaction timestamp.
df["time"] = pd.to_datetime(df["time"], errors="coerce")

# Numeric cleanup. We keep raw IDs as integers because later graph construction assumes that
# user_id and video_id can be used as node indices after offsetting item nodes by n_users.
for col in ["user_id", "video_id", "play_duration", "video_duration", "date", "timestamp", "watch_ratio"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

n_time_missing = int(df["time"].isna().sum())
if n_time_missing:
    warnings.warn(f"Dropping {n_time_missing:,} rows with unparseable interaction time")
    df = df[df["time"].notna()].copy()

id_missing = df[["user_id", "video_id"]].isna().any(axis=1)
if id_missing.any():
    warnings.warn(f"Dropping {int(id_missing.sum()):,} rows with missing user_id or video_id")
    df = df.loc[~id_missing].copy()

# Cast IDs after removing missing values.
df["user_id"] = df["user_id"].astype("int64")
df["video_id"] = df["video_id"].astype("int64")

print("raw rows kept:", f"{len(df):,}")
print("users:", f"{df['user_id'].nunique():,}")
print("videos:", f"{df['video_id'].nunique():,}")
display(df.head())


## 2. Bursts, Sessions, and Observed Engagement Labels

The raw data contain three dense activity bursts. We restrict to those bursts because later belief and structural estimation are defined burst-by-burst.

Session construction:

- Sort each user's interactions by time.
- Start a new session whenever the gap from the previous interaction exceeds one hour.
- Because the gaps between bursts are much larger than one hour, sessions naturally do not cross bursts.

Let an interaction be indexed by `(u, i, t)`, with realized play duration `d_uit` in milliseconds and realized watch ratio `w_uit`. The four observed heads are:

| Head | Mathematical definition | Interpretation | Leakage status |
|---|---|---|---|
| `y_complete` | `1{w_uit >= 1}` | User watched at least one full video length. | Outcome label only. |
| `y_long` | `1{d_uit >= 12000 or w_uit >= 1.5}` | Long watch either in absolute time or relative progress. | Outcome label only. |
| `y_rewatch` | `1{w_uit >= 2}` | User watched at least two video lengths, usually replay/loop behavior. | Outcome label only. |
| `y_neg` | `1{d_uit <= 2000}` | Very short watch, used as a negative response proxy. | Outcome label only. |

| Constructed feature | Description | Serving-safe? |
|---|---|---|
| `burst_id` | Dense-window indicator: 1, 2, or 3. | Yes. |
| `session` | Cumulative user session ID after applying the one-hour gap rule. | Yes. |
| `sess_rank` | Position of the current interaction within the session. | Yes if computed online from current session history. |
| `sess_len` | Final completed-session length. | No for live serving; useful for auditing and session-level summaries. |

These labels are observed after recommendation. They are saved because later notebooks need labels, but they should not be used as serving-time features.


In [ ]:
def assign_burst_id(frame, burst_ranges):
    out = frame.copy()
    interaction_date = out["time"].dt.normalize()
    burst_id = pd.Series(pd.NA, index=out.index, dtype="Int64")

    for idx, (start, end) in enumerate(burst_ranges, start=1):
        start_dt = pd.to_datetime(start)
        end_dt = pd.to_datetime(end)
        in_burst = interaction_date.between(start_dt, end_dt, inclusive="both")
        burst_id.loc[in_burst] = idx

    out["burst_id"] = burst_id
    out = out[out["burst_id"].notna()].copy()
    out["burst_id"] = out["burst_id"].astype("Int64")
    return out


def add_session_columns(frame, gap_seconds=3600):
    out = frame.sort_values(["user_id", "time", "video_id"], kind="mergesort").reset_index(drop=True)

    gap = (
        out.groupby("user_id", sort=False)["time"]
        .diff()
        .dt.total_seconds()
        .fillna(np.inf)
    )
    out["new_session"] = gap.gt(float(gap_seconds))

    # The first interaction for each user has gap=inf, so sessions start at 1.
    out["session"] = out.groupby("user_id", sort=False)["new_session"].cumsum().astype("Int64")
    out.drop(columns=["new_session"], inplace=True)

    out["sess_rank"] = (
        out.groupby(["user_id", "session"], sort=False).cumcount().astype("int32") + 1
    )
    out["sess_len"] = (
        out.groupby(["user_id", "session"], sort=False)["video_id"]
        .transform("size")
        .astype("int32")
    )
    return out


def add_observed_engagement_labels(frame):
    out = frame.copy()
    out["play_duration"] = pd.to_numeric(out["play_duration"], errors="coerce").fillna(0)
    out["watch_ratio"] = pd.to_numeric(out["watch_ratio"], errors="coerce").fillna(0.0)

    out["y_complete"] = out["watch_ratio"].ge(COMPLETE_WATCH_RATIO).astype("int8")
    out["y_long"] = (
        out["play_duration"].ge(LONG_PLAY_MS) | out["watch_ratio"].ge(LONG_WATCH_RATIO)
    ).astype("int8")
    out["y_rewatch"] = out["watch_ratio"].ge(REWATCH_WATCH_RATIO).astype("int8")
    out["y_neg"] = out["play_duration"].le(NEGATIVE_PLAY_MS).astype("int8")
    return out


df = assign_burst_id(df, BURST_RANGES)
df = add_session_columns(df, SESSION_GAP_SECONDS)
df = add_observed_engagement_labels(df)

print("rows inside bursts:", f"{len(df):,}")
print("burst counts:")
print(df["burst_id"].value_counts().sort_index())
print("label means:")
print(df[["y_complete", "y_long", "y_rewatch", "y_neg"]].mean().sort_index())

# Contract check: a (user_id, session) should not appear in multiple bursts.
session_burst_counts = (
    df[["user_id", "session", "burst_id"]]
    .drop_duplicates()
    .groupby(["user_id", "session"], observed=True)["burst_id"]
    .nunique()
)
if session_burst_counts.gt(1).any():
    raise ValueError("Some (user_id, session) pairs appear in multiple bursts; include burst_id in every downstream session key.")

print("sessions:", f"{df[['user_id', 'session']].drop_duplicates().shape[0]:,}")
display(df.head())


In [ ]:
# Lightweight descriptive diagnostics. These are helpful after rerunning the notebook
# but are not used by downstream code.
print("Sessions per user:")
display(df.groupby("user_id")["session"].nunique().describe())

print("Session length distribution:")
display(df.drop_duplicates(["user_id", "session"])["sess_len"].describe(percentiles=[0.5, 0.9, 0.95, 0.99]))

print("Watch ratio distribution:")
display(df["watch_ratio"].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]))


## 3. User Static Features

`user_features.csv` is one row per user. We prefix all columns with `u_` so later notebooks can tell user, item, context, and history features apart.

| Feature(s) | Raw source column(s) | Type after cleaning | Short description | Leakage note |
|---|---|---|---|---|
| `u_user_active_degree` | `user_active_degree` | category | Platform-defined activity segment. | Safe if the user snapshot predates serving. |
| `u_is_lowactive_period` | `is_lowactive_period` | int8 | Whether the user is in a low-activity period. | Safe if snapshot predates serving. |
| `u_is_live_streamer` | `is_live_streamer` | int8 | Whether the user is also a livestreamer. | Safe if snapshot predates serving. |
| `u_is_video_author` | `is_video_author` | int8 | Whether the user has uploaded videos. | Safe if snapshot predates serving. |
| `u_follow_user_num`, `u_fans_user_num`, `u_friend_user_num`, `u_register_days` | same names | float32 | Social/account counts and account age. | Safe if snapshot predates serving; watch snapshot timing in strict causal tests. |
| `u_follow_user_num_range`, `u_fans_user_num_range`, `u_friend_user_num_range`, `u_register_days_range` | same names | category | Binned versions of count variables. | Same as corresponding count. |
| `u_follow_user_num_log1p`, `u_fans_user_num_log1p`, `u_friend_user_num_log1p`, `u_register_days_log1p` | derived from numeric counts | float32 | Log-transformed count variables. | Same as corresponding count. |
| `u_onehot_feat0` to `u_onehot_feat17` | same names | category | Encrypted user segment features. | Safe if snapshot predates serving. |

Rules:

- Binary indicators are coerced to 0/1.
- Low-cardinality categorical variables are stored as pandas `category`.
- Count variables are stored as `float32`, and log1p versions are added.
- The merge is validated as many interactions to one user row.


In [ ]:
LOW_CARD_USER_CATS = [
    "user_active_degree",
    "follow_user_num_range",
    "fans_user_num_range",
    "friend_user_num_range",
    "register_days_range",
]
BINARY_USER_FLAGS = ["is_lowactive_period", "is_live_streamer", "is_video_author"]
NUMERIC_USER_COLS = ["follow_user_num", "fans_user_num", "friend_user_num", "register_days"]
ONEHOT_USER_COLS = [f"onehot_feat{i}" for i in range(18)]

user_header = pd.read_csv(USER_FEATURES_PATH, nrows=0).columns.tolist()
requested_user_cols = ["user_id"] + LOW_CARD_USER_CATS + BINARY_USER_FLAGS + NUMERIC_USER_COLS + ONEHOT_USER_COLS
use_user_cols = [c for c in requested_user_cols if c in user_header]
missing_user_cols = [c for c in requested_user_cols if c not in user_header]
if missing_user_cols:
    warnings.warn(f"User feature file is missing expected columns: {missing_user_cols}")

user_features = pd.read_csv(USER_FEATURES_PATH, usecols=use_user_cols, low_memory=False)
user_features = user_features.drop_duplicates(subset=["user_id"]).copy()
user_features["user_id"] = pd.to_numeric(user_features["user_id"], errors="coerce").astype("Int64")
user_features = user_features[user_features["user_id"].notna()].copy()
user_features["user_id"] = user_features["user_id"].astype("int64")

for col in BINARY_USER_FLAGS:
    if col in user_features.columns:
        user_features[col] = (
            pd.to_numeric(user_features[col], errors="coerce")
            .fillna(0)
            .clip(0, 1)
            .astype("int8")
        )

for col in ONEHOT_USER_COLS:
    if col in user_features.columns:
        user_features[col] = user_features[col].astype("string").fillna("unknown").replace({"": "unknown"}).astype("category")

for col in LOW_CARD_USER_CATS:
    if col in user_features.columns:
        user_features[col] = user_features[col].astype("string").fillna("unknown").replace({"": "unknown"}).astype("category")

for col in NUMERIC_USER_COLS:
    if col in user_features.columns:
        vals = pd.to_numeric(user_features[col], errors="coerce")
        user_features[col] = vals.astype("float32")
        user_features[f"{col}_log1p"] = np.log1p(vals.clip(lower=0)).astype("float32")

user_features = user_features.rename(columns={c: f"u_{c}" for c in user_features.columns if c != "user_id"})

n_before = len(df)
df = df.merge(user_features, on="user_id", how="left", validate="many_to_one")
assert len(df) == n_before

u_cols = [c for c in df.columns if c.startswith("u_")]
print("merged user features:", len(u_cols))
print("users without a static-feature row:", f"{df.loc[df[u_cols].isna().all(axis=1), 'user_id'].nunique():,}")
display(df[u_cols].head())


## 4. Item Static Features and Category Metadata

`item_daily_features.csv` has one row per video per date. The file also contains many daily aggregate behavior counters. This notebook keeps only stable item attributes and item-age variables; it deliberately excludes daily aggregate counters because those counters summarize realized behavior and can leak same-day or future response information.

| Feature(s) | Raw source column(s) | Type after cleaning | Short description | Leakage note |
|---|---|---|---|---|
| `i_author_id` | `author_id` | category | Video creator ID. | Safe as item metadata. |
| `i_video_type` | `video_type` | category | Video type, such as normal/ad. | Safe as item metadata. |
| `i_upload_type` | `upload_type` | category | Upload source or method. | Safe as item metadata. |
| `i_visible_status` | `visible_status` | category | Visibility status in the app. | Potentially snapshot-like; safe for descriptive work, but check timing in strict historical serving tests. |
| `i_music_id` | `music_id` | category | Background music ID. | Safe as item metadata. |
| `i_video_tag_id`, `i_video_tag_name` | `video_tag_id`, `video_tag_name` | category | Platform tag ID and name. | Safe as item metadata. |
| `i_video_duration` | `video_duration` | float32 | Video length in milliseconds. | Safe; item property. |
| `i_aspect_ratio` | `video_width / video_height` | float32 | Video width divided by height. | Safe; item property. |
| `i_age_since_upload_days` | `time - upload_dt` | float32 | Video age at the interaction time. | Safe; uses current time and upload date. |
| `i_cat_level1_id/name`, `i_cat_level2_id/name`, `i_cat_level3_id/name` | `kuairec_caption_category.csv` | numeric/string | Caption-derived content category hierarchy. | Safe as item metadata. |

Important: the raw interaction table has a `video_duration` column. We keep it temporarily as `raw_video_duration`, merge item metadata as `i_video_duration`, and use the raw duration only as a fallback when item metadata is missing.

Excluded on purpose: `show_cnt`, `show_user_num`, `play_cnt`, `play_user_num`, `complete_play_cnt`, `valid_play_cnt`, `long_time_play_cnt`, `short_time_play_cnt`, `play_progress`, `like_cnt`, `comment_cnt`, `follow_cnt`, `share_cnt`, `download_cnt`, `report_cnt`, `reduce_similar_cnt`, `collect_cnt`, and the corresponding user-count fields.


In [ ]:
# Preserve the interaction-table duration before merging item metadata.
if "video_duration" in df.columns:
    df = df.rename(columns={"video_duration": "raw_video_duration"})

item_header = pd.read_csv(ITEM_DAILY_FEATURES_PATH, nrows=0).columns.tolist()
item_candidate_cols = [
    "video_id", "date", "author_id", "video_type", "upload_dt", "upload_type",
    "visible_status", "video_duration", "video_width", "video_height",
    "music_id", "video_tag_id", "video_tag_name",
]
use_item_cols = [c for c in item_candidate_cols if c in item_header]
item_daily = pd.read_csv(ITEM_DAILY_FEATURES_PATH, usecols=use_item_cols, low_memory=False)

item_daily["video_id"] = pd.to_numeric(item_daily["video_id"], errors="coerce").astype("Int64")
item_daily = item_daily[item_daily["video_id"].notna()].copy()
item_daily["video_id"] = item_daily["video_id"].astype("int64")

# Sort by date so the first row per video is deterministic. Static attributes should agree
# across dates; if not, this choice keeps the earliest observed metadata row.
if "date" in item_daily.columns:
    item_daily["date"] = pd.to_numeric(item_daily["date"], errors="coerce")
    item_daily = item_daily.sort_values(["video_id", "date"], kind="mergesort")
else:
    item_daily = item_daily.sort_values(["video_id"], kind="mergesort")

per_video = item_daily.drop_duplicates(subset=["video_id"], keep="first").copy()

# Upload date lets us compute item age at interaction time.
if "upload_dt" in per_video.columns:
    per_video["i_upload_dt"] = pd.to_datetime(per_video["upload_dt"], errors="coerce")
else:
    per_video["i_upload_dt"] = pd.NaT

if {"video_width", "video_height"}.issubset(per_video.columns):
    width = pd.to_numeric(per_video["video_width"], errors="coerce")
    height = pd.to_numeric(per_video["video_height"], errors="coerce").replace(0, np.nan)
    per_video["i_aspect_ratio"] = (width / height).astype("float32")
else:
    per_video["i_aspect_ratio"] = np.nan

item_cat_cols = ["author_id", "video_type", "upload_type", "visible_status", "music_id", "video_tag_id", "video_tag_name"]
for col in item_cat_cols:
    if col in per_video.columns:
        per_video[f"i_{col}"] = per_video[col].astype("string").fillna("unknown").replace({"": "unknown"}).astype("category")

if "video_duration" in per_video.columns:
    per_video["i_video_duration"] = pd.to_numeric(per_video["video_duration"], errors="coerce").astype("float32")
else:
    per_video["i_video_duration"] = np.nan

keep_cols = ["video_id", "i_aspect_ratio", "i_upload_dt", "i_video_duration"]
keep_cols += [f"i_{c}" for c in item_cat_cols if f"i_{c}" in per_video.columns]
per_video = per_video[keep_cols].copy()

n_before = len(df)
df = df.merge(per_video, on="video_id", how="left", validate="many_to_one")
assert len(df) == n_before

# Fill item duration from raw interaction duration if item metadata lacks it.
if "raw_video_duration" in df.columns:
    df["i_video_duration"] = df["i_video_duration"].fillna(pd.to_numeric(df["raw_video_duration"], errors="coerce"))
    df = df.drop(columns=["raw_video_duration"])

df["i_age_since_upload_days"] = (
    (df["time"] - df["i_upload_dt"]).dt.total_seconds() / (24 * 3600)
).astype("float32")
df = df.drop(columns=["i_upload_dt"])

print("item static feature columns:", [c for c in df.columns if c.startswith("i_")][:20])
print("interactions with missing item duration:", int(df["i_video_duration"].isna().sum()))
display(df[["video_id", "i_video_duration", "i_age_since_upload_days", "i_aspect_ratio"]].head())


In [ ]:
# Category metadata from captions/topics.
category_cols = [
    "video_id",
    "first_level_category_id", "first_level_category_name",
    "second_level_category_id", "second_level_category_name",
    "third_level_category_id", "third_level_category_name",
]
caption_categories = pd.read_csv(
    CAPTION_CATEGORY_PATH,
    usecols=category_cols,
    engine="python",
    encoding="utf-8",
    on_bad_lines="warn",
)
caption_categories["video_id"] = pd.to_numeric(caption_categories["video_id"], errors="coerce").astype("Int64")
caption_categories = caption_categories[caption_categories["video_id"].notna()].copy()
caption_categories["video_id"] = caption_categories["video_id"].astype("int64")
caption_categories = caption_categories.drop_duplicates(subset=["video_id"], keep="first")

caption_categories = caption_categories.rename(columns={
    "first_level_category_id": "i_cat_level1_id",
    "first_level_category_name": "i_cat_level1_name",
    "second_level_category_id": "i_cat_level2_id",
    "second_level_category_name": "i_cat_level2_name",
    "third_level_category_id": "i_cat_level3_id",
    "third_level_category_name": "i_cat_level3_name",
})

n_before = len(df)
df = df.merge(caption_categories, on="video_id", how="left", validate="many_to_one")
assert len(df) == n_before

# The UNKNOWN category is important later because structural utility is normalized relative
# to this category in the preference-estimation notebook.
df["i_cat_level1_id"] = pd.to_numeric(df["i_cat_level1_id"], errors="coerce").fillna(UNKNOWN_LEVEL1_CATEGORY_ID).astype("int64")
df["i_cat_level1_name"] = df["i_cat_level1_name"].astype("string").fillna(UNKNOWN_CATEGORY_NAME).replace({"": UNKNOWN_CATEGORY_NAME})

for col in ["i_cat_level2_id", "i_cat_level3_id"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

for col in ["i_cat_level2_name", "i_cat_level3_name"]:
    df[col] = df[col].astype("string").fillna(UNKNOWN_CATEGORY_NAME).replace({"": UNKNOWN_CATEGORY_NAME})

category_id_table = (
    df.groupby(["i_cat_level1_id", "i_cat_level1_name"], dropna=False)
    .agg(
        n_videos=("video_id", "nunique"),
        n_interactions=("video_id", "size"),
    )
    .reset_index()
    .sort_values(["i_cat_level1_id", "i_cat_level1_name"], kind="mergesort")
    .reset_index(drop=True)
)

print("level-1 category count:", len(category_id_table))
print("UNKNOWN level-1 interactions:", int(df["i_cat_level1_id"].eq(UNKNOWN_LEVEL1_CATEGORY_ID).sum()))
print("Full level-1 category-id table. Expected support for the structural model: 39 categories.")
display(category_id_table)
display(df[["video_id", "i_cat_level1_id", "i_cat_level1_name", "i_cat_level2_id", "i_cat_level3_id"]].head())


## 5. Context Features

Context features describe the interaction environment and are available before observing the current response.

| Feature | Construction | Short description | Leakage note |
|---|---|---|---|
| `ctx_hour_sin` | `sin(2*pi*seconds_since_midnight/86400)` | Cyclical time-of-day encoding. | Safe; uses recommendation time. |
| `ctx_hour_cos` | `cos(2*pi*seconds_since_midnight/86400)` | Companion cyclical time-of-day encoding. | Safe; uses recommendation time. |
| `ctx_is_weekend` | `1{dayofweek in {Saturday, Sunday}}` | Weekend indicator. | Safe; uses recommendation date. |

I derive time-of-day and weekend from `time`, the parsed interaction timestamp. The previous notebook used `pd.to_datetime(timestamp, unit="s")`; that often works because `timestamp` is Unix time, but using `time` keeps the transformation transparent.


In [ ]:
seconds_since_midnight = (
    df["time"].dt.hour * 3600
    + df["time"].dt.minute * 60
    + df["time"].dt.second
).astype("float32")
frac_day = seconds_since_midnight / np.float32(24 * 3600)

df["ctx_hour_sin"] = np.sin(2 * np.pi * frac_day).astype("float32")
df["ctx_hour_cos"] = np.cos(2 * np.pi * frac_day).astype("float32")
df["ctx_is_weekend"] = df["time"].dt.dayofweek.isin([5, 6]).astype("int8")

print("weekend share:", float(df["ctx_is_weekend"].mean()))
display(df[["time", "ctx_hour_sin", "ctx_hour_cos", "ctx_is_weekend"]].head())


## 6. History Features

History features must not use the current interaction's outcome. This section uses two patterns:

1. Compute a cumulative or exponentially weighted statistic including the current row.
2. Shift it by one interaction within the relevant group.

This gives the value that would have been known immediately before the current recommendation.

| Feature(s) | Construction | Short description | Leakage note |
|---|---|---|---|
| `hist_ema_y_complete`, `hist_ema_y_long`, `hist_ema_y_rewatch`, `hist_ema_y_neg` | Per-user EWMA of the corresponding observed head, shifted by one interaction. | User's previous response tendency. | Safe; current outcome is excluded. |
| `hist_ema_watchratio` | Per-user EWMA of realized `watch_ratio`, shifted by one interaction. | User's prior watch-ratio tendency. | Safe; current outcome is excluded. |
| `hist_cat_ema_complete` | Per-user x level-1-category EWMA of `y_complete`, shifted by one interaction within that cell. | User's previous completion tendency for similar broad content. | Safe; current outcome is excluded. |
| `hist_cat_entropy_l2` | Entropy of the user's previous level-2 category history. | Prior diversity of consumed content. | Safe; current category is counted only after the row's entropy is computed. |
| `hist_author_recency_days` | Days since previous exposure to the same author within the same burst. | Author recency. | Safe; uses previous author exposure only. |
| `hist_last_complete_author` | Previous `y_complete` for the same user-author pair within the same burst. | Whether the last same-author exposure was completed. | Safe; shifted by one author interaction. |
| `hist_has_author_history` | Indicator that previous same-author exposure exists in the current burst. | Whether the author is familiar to the user in this burst. | Safe; uses previous author exposure only. |
| `hist_prev_sess_len` | Length of the previous completed session in the same burst. | Prior session intensity. | Safe; previous session is already completed. |
| `hist_intersess_gap_h` | Hours between previous session end and current session start. | Time away since last session. | Safe; previous session is already completed. |

Leakage audit: every feature in this section is either shifted by one interaction, computed from previous completed sessions, or computed from previous same-author interactions. None of these history variables uses the current row's realized `play_duration`, `watch_ratio`, or `y_*` value.


In [ ]:
# Sort once before history construction. All following cumulative features rely on this order.
df = df.sort_values(["user_id", "time", "video_id"], kind="mergesort").reset_index(drop=True)

alpha = 0.1
outcome_history_cols = ["y_complete", "y_long", "y_rewatch", "y_neg", "watch_ratio"]

for col in outcome_history_cols:
    ema_including_current = df.groupby("user_id", sort=False)[col].transform(
        lambda s: s.ewm(alpha=alpha, adjust=False).mean()
    )
    suffix = col if col != "watch_ratio" else "watchratio"
    hist_col = f"hist_ema_{suffix}"
    df[hist_col] = ema_including_current.groupby(df["user_id"], sort=False).shift(1).astype("float32")

# User x top-level-category EMA for completion. This only uses previous completion outcomes
# within the same user/category cell.
cat_col = "i_cat_level1_id"
cat_ema_including_current = df.groupby(["user_id", cat_col], sort=False)["y_complete"].transform(
    lambda s: s.ewm(alpha=alpha, adjust=False).mean()
)
df["hist_cat_ema_complete"] = (
    cat_ema_including_current
    .groupby([df["user_id"], df[cat_col]], sort=False)
    .shift(1)
    .astype("float32")
)

history_ema_cols = [c for c in df.columns if c.startswith("hist_ema_")] + ["hist_cat_ema_complete"]
print("EMA history columns:", history_ema_cols)
display(df[["user_id", "time", "y_complete", "hist_ema_y_complete", "i_cat_level1_id", "hist_cat_ema_complete"]].head(10))


In [ ]:
def entropy_from_counts(counts):
    counts = np.asarray(counts, dtype="float64")
    total = counts.sum()
    if total <= 0:
        return 0.0
    p = counts / total
    p = p[p > 0]
    return float(-(p * np.log(p)).sum())


def historical_entropy_for_series(values):
    """Return entropy of previous values before each row in a user history."""
    counts = {}
    out = np.zeros(len(values), dtype="float32")

    for pos, value in enumerate(values):
        # Entropy before observing current value.
        out[pos] = entropy_from_counts(list(counts.values()))

        if pd.notna(value):
            key = int(value)
            counts[key] = counts.get(key, 0) + 1

    return out


def add_historical_level2_entropy(frame):
    pieces = []
    for _, group in frame.groupby("user_id", sort=False):
        values = group["i_cat_level2_id"].to_numpy()
        pieces.append(pd.Series(historical_entropy_for_series(values), index=group.index))
    return pd.concat(pieces).sort_index().astype("float32")

# This can take a little time on the full table, but it is deliberately explicit.
df["hist_cat_entropy_l2"] = add_historical_level2_entropy(df)

print("historical L2 entropy summary:")
display(df["hist_cat_entropy_l2"].describe())


In [ ]:
# Author-history features are computed within each burst because long gaps between bursts
# should reset the local author-recency environment.
df = df.sort_values(["user_id", "burst_id", "time", "video_id"], kind="mergesort").reset_index(drop=True)

if "i_author_id" not in df.columns:
    warnings.warn("i_author_id is missing; author-history features will be all missing/zero")
    df["hist_author_recency_days"] = np.nan
    df["hist_last_complete_author"] = np.nan
    df["hist_has_author_history"] = np.int8(0)
else:
    previous_author_time = df.groupby(["user_id", "burst_id", "i_author_id"], observed=True, sort=False)["time"].shift(1)
    df["hist_author_recency_days"] = (
        (df["time"] - previous_author_time).dt.total_seconds() / (24 * 3600)
    ).astype("float32")

    df["hist_last_complete_author"] = (
        df.groupby(["user_id", "burst_id", "i_author_id"], observed=True, sort=False)["y_complete"]
        .shift(1)
        .astype("float32")
    )
    df["hist_has_author_history"] = previous_author_time.notna().astype("int8")

print("author-history summary:")
display(df[["hist_author_recency_days", "hist_last_complete_author", "hist_has_author_history"]].describe(include="all"))


In [ ]:
# Previous-session features are constant for every row in a session.
df = df.sort_values(["user_id", "burst_id", "session", "time", "video_id"], kind="mergesort").reset_index(drop=True)

session_level = (
    df.groupby(["user_id", "burst_id", "session"], observed=True, sort=False)
    .agg(
        sess_start=("time", "min"),
        sess_end=("time", "max"),
        sess_len=("sess_len", "first"),
    )
    .reset_index()
    .sort_values(["user_id", "burst_id", "sess_start"], kind="mergesort")
)

session_level["hist_prev_sess_len"] = (
    session_level.groupby(["user_id", "burst_id"], observed=True, sort=False)["sess_len"].shift(1)
)
previous_session_end = session_level.groupby(["user_id", "burst_id"], observed=True, sort=False)["sess_end"].shift(1)
session_level["hist_intersess_gap_h"] = (
    (session_level["sess_start"] - previous_session_end).dt.total_seconds() / 3600.0
)

session_history = session_level[["user_id", "burst_id", "session", "hist_prev_sess_len", "hist_intersess_gap_h"]]

n_before = len(df)
df = df.merge(session_history, on=["user_id", "burst_id", "session"], how="left", validate="many_to_one")
assert len(df) == n_before

df["hist_prev_sess_len"] = df["hist_prev_sess_len"].fillna(0).astype("float32")
df["hist_intersess_gap_h"] = df["hist_intersess_gap_h"].fillna(0).astype("float32")

print("previous-session feature summary:")
display(df[["hist_prev_sess_len", "hist_intersess_gap_h"]].describe(percentiles=[0.5, 0.9, 0.95, 0.99]))


## 7. Final Validation

These checks are meant to catch silent mistakes before a downstream notebook uses the table.

Validation goals:

| Check | Why it matters |
|---|---|
| Required columns exist | Prevent downstream notebooks from silently using a partial table. |
| IDs and timestamps are nonmissing | Keys and chronological ordering must be valid. |
| `sess_rank` starts at 1 | Confirms session ordering is coherent. |
| `sess_len` matches grouped session size | Confirms the session construction did not drift after merges. |
| UNKNOWN level-1 category exists | Later structural utility normalizes one category relative to UNKNOWN. |
| Context trigonometric features are finite | Catches bad timestamps before saving. |
| Historical entropy is nonnegative | Catches invalid history construction. |

Leakage reminder: the validation table includes outcome columns and `sess_len`, but the existence of a column is not permission to use it as a serving-time feature.


In [ ]:
# Put rows back into canonical chronological order.
df = df.sort_values(["user_id", "time", "video_id"], kind="mergesort").reset_index(drop=True)

required_final_cols = [
    "user_id", "video_id", "play_duration", "i_video_duration", "time", "date", "timestamp", "watch_ratio",
    "burst_id", "session", "sess_rank", "sess_len",
    "y_complete", "y_long", "y_rewatch", "y_neg",
    "ctx_hour_sin", "ctx_hour_cos", "ctx_is_weekend",
    "hist_ema_y_complete", "hist_ema_y_long", "hist_ema_y_rewatch", "hist_ema_y_neg", "hist_ema_watchratio",
    "hist_cat_ema_complete", "hist_cat_entropy_l2",
    "hist_author_recency_days", "hist_last_complete_author", "hist_has_author_history",
    "hist_prev_sess_len", "hist_intersess_gap_h",
    "i_cat_level1_id", "i_cat_level1_name",
]
missing_final = [c for c in required_final_cols if c not in df.columns]
if missing_final:
    raise KeyError(f"Final table is missing required columns: {missing_final}")

checks = {
    "no_missing_user_id": df["user_id"].notna().all(),
    "no_missing_video_id": df["video_id"].notna().all(),
    "no_missing_time": df["time"].notna().all(),
    "session_rank_min_is_one": int(df["sess_rank"].min()) == 1,
    "session_len_matches_group_size": bool(
        df.groupby(["user_id", "session"], observed=True)["video_id"].transform("size").eq(df["sess_len"]).all()
    ),
    "level1_unknown_present": bool(df["i_cat_level1_id"].eq(UNKNOWN_LEVEL1_CATEGORY_ID).any()),
    "ctx_hour_sin_finite": np.isfinite(df["ctx_hour_sin"]).all(),
    "ctx_hour_cos_finite": np.isfinite(df["ctx_hour_cos"]).all(),
    "history_entropy_nonnegative": df["hist_cat_entropy_l2"].ge(0).all(),
}

for name, ok in checks.items():
    print(f"{name}: {ok}")

failed = [name for name, ok in checks.items() if not ok]
if failed:
    raise ValueError(f"Final validation failed: {failed}")

print("final shape:", df.shape)
print("final column count:", len(df.columns))
print("users:", df["user_id"].nunique(), "videos:", df["video_id"].nunique())
print("sessions:", df[["user_id", "session"]].drop_duplicates().shape[0])


In [ ]:
# Optional: inspect final dtypes and a small preview before writing.
display(df.dtypes.to_frame("dtype").T)
display(df.head())


## 8. Save

The parquet file is the canonical artifact. It preserves dtypes better than CSV and is much faster for downstream notebooks.

| Output | Role |
|---|---|
| `processed_data.parquet` | Canonical downstream artifact. |
| `processed_data.csv` | Optional compatibility export, disabled by default. |

I keep CSV writing behind a flag because the file is large and easy to create accidentally.


In [ ]:
df.to_parquet(OUTPUT_PARQUET, index=False)
print("saved parquet:", OUTPUT_PARQUET)

if WRITE_CSV:
    df.to_csv(OUTPUT_CSV, index=False)
    print("saved csv:", OUTPUT_CSV)
else:
    print("WRITE_CSV is False; skipped CSV export")


## Notes for Future Notebooks

- `y_*` columns are realized observed outcomes.
- Predicted value-model heads should be named `yhat_*` and saved in a separate artifact or added by the GNN prediction notebook.
- Session-level keys should generally include `burst_id`, `user_id`, and `session`, even though the current construction makes `(user_id, session)` unique across bursts.
- History features are serving-time features; they are shifted to exclude the current interaction.
- `sess_len` is not serving-safe during a live session. Prefer `sess_rank`, `hist_prev_sess_len`, and `hist_intersess_gap_h` for inference-time models.
- `i_cat_level1_id = -124` is the UNKNOWN category and is used later as the structural utility reference category.
- Daily item aggregates from `item_daily_features.csv` remain excluded unless a future notebook can prove correct time alignment.
